# Phase 1.1 — NASS Yield Data Ingestion

Fetch county-level corn yield history (2005–2024) for all 5 target states from the USDA NASS QuickStats API and cache the results to S3.

**States**: Iowa (IA), Colorado (CO), Wisconsin (WI), Missouri (MO), Nebraska (NE)  
**Metric**: CORN, GRAIN — YIELD, MEASURED IN BU/ACRE


In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
# If running from notebooks/ directory:
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from data_utils import fetch_all_states, load_all_states, STATES

BUCKET = "cornsight-data"


## Step 1 — Fetch from NASS API and cache to S3

Run this once. After the CSVs are in S3, use `load_all_states()` instead to avoid re-hitting the API.


In [ ]:
# NOTE: Set your NASS API key before running
# Either: os.environ["NASS_API_KEY"] = "your_key_here"
# Or set it in your shell: $env:NASS_API_KEY = "your_key"

yields_df = fetch_all_states(BUCKET, year_start=2005, year_end=2024)
print(f"\nTotal rows fetched: {len(yields_df)}")
print(yields_df["state"].value_counts())


## Step 2 — Reload from S3 (skip the API on subsequent runs)


In [ ]:
yields_df = load_all_states(BUCKET)
print(f"Loaded {len(yields_df)} rows from S3")
yields_df.head()


## Step 3 — Quick sanity checks


In [ ]:
import matplotlib.pyplot as plt

# Check year coverage per state
print("Year range per state:")
print(yields_df.groupby("state")["year"].agg(["min", "max", "count"]))

# Check for missing yield values
missing = yields_df["yield_bu_acre"].isna().sum()
print(f"\nMissing yield values: {missing} / {len(yields_df)}")

# State-level average yield over time
state_avg = (
    yields_df.groupby(["state", "year"])["yield_bu_acre"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for state, grp in state_avg.groupby("state"):
    ax.plot(grp["year"].astype(int), grp["yield_bu_acre"], marker="o", label=state)
ax.set_title("State-Average Corn Yield (bu/acre) — 2005–2024")
ax.set_xlabel("Year")
ax.set_ylabel("bu/acre")
ax.legend()
plt.tight_layout()
plt.show()
